In [0]:
!pip install -r ../requirements.txt
%restart_python

In [0]:
import pandas as pd

csv_path = "/Volumes/abs_data/default/raw_volume/Emails.csv"
df = pd.read_csv(csv_path)

f_df = df[df["ExtractedBodyText"].str.len() >= 800]

def get_value(row_index):
    return f_df.iloc[row_index]["ExtractedBodyText"]


In [0]:
from shared.llm_utils import llm_basic, llm_structured, format_system_prompt
from pydantic import BaseModel

class EmailSummary(BaseModel):
    summary: str
    search_phrases: list[str]
    persons_of_interests: list[str]

temp = """ \
You are provided an email to analyse and your task is to extract the following details:
 - detailed summary of the email
 - list of general search_phrases and additional versions of the search phrases which will be used for categorising the email
 - persons and organisations of interests that were noted in the email
output will just include {summary: str, search_phrases: list[str],  persons_of_interests: list[str]}
    """

# print(format_system_prompt(temp, model="gpt-5-mini"))

In [0]:
ROW_NUMBER = 33

value = get_value(ROW_NUMBER)
print(value)

In [0]:
EMAIL_PROMPT = """ \
<system_role>
    You are an expert Email Analyst. Use a concise, formal, and factual tone. Prioritize accuracy, normalization, and brevity.
</system_role>

<instructions>
    You will be given the raw text of an email. Your task is to extract three items and output exactly one valid JSON object (no surrounding text, no explanations, no commentary):

    - summary: A single string containing a detailed summary of the email (purpose, key points, actions requested, deadlines/dates, attachments or references, and tone).
    - search_phrases: A list of strings containing general search phrases and additional phrase variations useful for categorizing or searching for similar emails. Include synonyms, abbreviations, keyword pairs, and likely alternate phrasings.
    - persons_of_interests: A list of strings containing both persons and organisations mentioned or implied in the email. Normalize names (e.g., "John Smith", "Acme Corp"), deduplicate, and include role/relationship only if explicitly present in the email.

    Rules:
    - Output only one valid JSON object with exactly these three keys: { "summary": str, "search_phrases": [str], "persons_of_interests": [str] }.
    - Do not output any other text, commentary, or explanatory content.
    - Ensure the JSON is syntactically valid (use double quotes for keys and strings).
    - Keep the summary concise but detailed (typically 2–6 sentences).
    - Provide at least 5 search_phrases where applicable, and include variations (singular/plural, synonyms, abbreviations).
    - For persons_of_interests include both individual people and organisations. If none are present, return an empty list [].
</instructions>
    """

summarise = llm_structured(user_prompt=value, system_prompt=EMAIL_PROMPT, text_format=EmailSummary)
display(summarise)